In [1]:
!rm -rf /content/data



In [2]:
!pip install python-Levenshtein

In [3]:
!apt-get update && apt-get install -y poppler-utils
!pip install -q "transformers<4.40.0" pdf2image accelerate

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [4]:
import os
from pathlib import Path
import torch
from PIL import Image
from pdf2image import convert_from_path
from transformers import NougatProcessor, VisionEncoderDecoderModel

# --- CONFIGURATION ---
PDF_INPUT_DIR = Path("/content/drive/MyDrive/raw_pdfs")
MD_OUTPUT_DIR = Path("./data/markdown_sections")
MD_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Check for GPU acceleration in Colab
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] Utilizing device footprint: {device}")

# 2. Load the Nougat Processor and Model
print("[*] Downloading and loading Meta Nougat model...")
processor = NougatProcessor.from_pretrained("facebook/nougat-small")
model = VisionEncoderDecoderModel.from_pretrained("facebook/nougat-small")
model.to(device)

def process_pdf_with_nougat(pdf_path: Path):
    print(f"\n[*] Slicing PDF: {pdf_path.name}")

    # Convert PDF pages into PIL Images for the vision encoder
    try:
        images = convert_from_path(pdf_path, dpi=150)
    except Exception as e:
        print(f"[-] Error converting PDF pages: {e}")
        return

    full_markdown_text = ""

    for page_num, image in enumerate(images, 1):
        print(f"    -> Processing page {page_num}/{len(images)}...")

        # Prepare image pixel values for the transformer
        pixel_values = processor(images=image, return_tensors="pt").pixel_values

        # Generate the text stream matching the visual document layout
        with torch.no_grad():
            outputs = model.generate(
                pixel_values.to(device),
                min_length=1,
                max_new_tokens=1024,
                bad_words_ids=[[processor.tokenizer.unk_token_id]],
            )

        # Decode output tokens back into clean text
        page_sequence = processor.batch_decode(outputs, skip_special_tokens=True)[0]
        page_markdown = processor.post_process_generation(page_sequence)
        full_markdown_text += page_markdown + "\n\n"

    # Save out raw structured markdown text artifact
    output_file = MD_OUTPUT_DIR / f"{pdf_path.stem}.md"
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(full_markdown_text)
    print(f"[+] Saved parsed structural artifact to: {output_file}")

# --- EXECUTION ---
pdf_files = list(PDF_INPUT_DIR.glob("*.pdf"))
if not pdf_files:
    print(f"[-] No PDFs found in {PDF_INPUT_DIR}. Please upload files first.")
else:
    for pdf in pdf_files:
        process_pdf_with_nougat(pdf)
    print("\n[+] All PDF artifacts successfully structured using Nougat!")

[*] Utilizing device footprint: cuda
[*] Downloading and loading Meta Nougat model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



[*] Slicing PDF: 2505.14726v1.pdf
    -> Processing page 1/8...
    -> Processing page 2/8...
    -> Processing page 3/8...
    -> Processing page 4/8...
    -> Processing page 5/8...
    -> Processing page 6/8...
    -> Processing page 7/8...
    -> Processing page 8/8...
[+] Saved parsed structural artifact to: data/markdown_sections/2505.14726v1.md

[*] Slicing PDF: 2603.00529v1.pdf
    -> Processing page 1/17...
    -> Processing page 2/17...
    -> Processing page 3/17...
    -> Processing page 4/17...
    -> Processing page 5/17...
    -> Processing page 6/17...
    -> Processing page 7/17...
    -> Processing page 8/17...
    -> Processing page 9/17...
    -> Processing page 10/17...
    -> Processing page 11/17...
    -> Processing page 12/17...
    -> Processing page 13/17...
    -> Processing page 14/17...
    -> Processing page 15/17...
    -> Processing page 16/17...
    -> Processing page 17/17...
[+] Saved parsed structural artifact to: data/markdown_sections/2603.00529v